# AllnAction RoboTaxi — Auto-Prelabel 11 Lớp (YOLO-World + SAM 2.1 Base — KHÔNG CẦN TOKEN)

Pipeline **0 cần HuggingFace Token**, chạy ngay trên Colab T4:
1. **YOLO-World (`yolov8l-worldv2.pt`):** Bắt Bounding Box theo text prompt cho cả 11 lớp (gồm `auto rickshaw`, `tricycle`, `cyclo`, `tuk tuk`, `trailer` → `3: Other vehicle`).
2. **SAM 2.1 Base (`sam2.1_b.pt`):** Nhận Bounding Box làm prompt để cắt mask polygon sắc nét.
3. Tự gộp `person` + `motorcycle/bicycle` → `Motorcyclist`/`Cyclist`, lọc mask `< 65 px`, xuất `sam2_prelabeled.zip`.

In [ ]:
#@title 1. Cài đặt Ultralytics & kiểm tra GPU
!pip install -qU ultralytics opencv-python pyyaml
import torch
print("CUDA available:", torch.cuda.is_available(), "| Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
#@title 2. Kéo repo & tải ~60 ảnh Other vehicle + đường phố VN
!git clone -q https://github.com/tiennguyeniotd22-ops/allnaction-traffic-seg.git /content/repo || (cd /content/repo && git pull -q)
!python /content/repo/scripts/download_other_vehicle_images.py --out /content/raw_other_vehicle --per-query 12
!ls -lh /content/raw_other_vehicle | head -n 15

In [ ]:
#@title 3. Chạy YOLO-World + SAM 2.1 Base Prelabel 11 lớp (Không cần token)
import shutil
from collections import Counter
from pathlib import Path
import cv2
import numpy as np
import yaml
from ultralytics import YOLOWorld, SAM

IMAGES_DIR = "/content/raw_other_vehicle"  #@param {type:"string"}
CONF_THRES = 0.20  #@param {type:"number"}
SAM_WEIGHTS = "sam2.1_b.pt"  #@param ["sam2.1_t.pt", "sam2.1_s.pt", "sam2.1_b.pt", "sam2.1_l.pt"]

CLASSES = {
    0: "Car", 1: "Bus", 2: "Truck", 3: "Other vehicle",
    4: "Motorcycle", 5: "Motorcyclist", 6: "Bicycle", 7: "Cyclist",
    8: "Pedestrian", 9: "Sign", 10: "Traffic light",
}

PROMPT_GROUPS = [
    ("car", 0), ("bus", 1), ("truck", 2),
    ("auto rickshaw", 3), ("tricycle", 3), ("cyclo", 3), ("tuk tuk", 3), ("trailer", 3),
    ("motorcycle", 4), ("bicycle", 6), ("person", 8),
    ("traffic sign", 9), ("traffic light", 10),
]

def mask_to_norm_poly(mask_bin: np.ndarray, w: int, h: int, min_area_640: float = 65.0) -> np.ndarray | None:
    contours, _ = cv2.findContours(mask_bin.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    largest = max(contours, key=cv2.contourArea)
    peri = cv2.arcLength(largest, True)
    approx = cv2.approxPolyDP(largest, 0.002 * peri, True).reshape(-1, 2).astype(np.float32)
    if len(approx) < 3:
        return None
    scale = 640.0 / float(max(w, h))
    if cv2.contourArea(approx * scale) < min_area_640:
        return None
    approx[:, 0] = np.clip(approx[:, 0] / float(w), 0.0, 1.0)
    approx[:, 1] = np.clip(approx[:, 1] / float(h), 0.0, 1.0)
    return approx

def masks_overlap(m_person: np.ndarray, m_bike: np.ndarray) -> bool:
    if np.logical_and(m_person > 0, m_bike > 0).sum() > 0:
        return True
    dil = cv2.dilate(m_bike.astype(np.uint8), np.ones((9, 9), np.uint8))
    return np.logical_and(m_person > 0, dil > 0).sum() >= 30

det_model = YOLOWorld("yolov8l-worldv2.pt")
det_model.set_classes([p[0] for p in PROMPT_GROUPS])
sam_model = SAM(SAM_WEIGHTS)

out_dir = Path("/content/sam2_prelabeled")
img_out = out_dir / "images" / "train"
lbl_out = out_dir / "labels" / "train"
img_out.mkdir(parents=True, exist_ok=True)
lbl_out.mkdir(parents=True, exist_ok=True)

with open(out_dir / "data.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump({"path": ".", "train": "images/train", "val": "images/train", "nc": 11, "names": CLASSES}, f, sort_keys=False, allow_unicode=True)

img_paths = sorted(p for p in Path(IMAGES_DIR).glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
stats = Counter()

for idx, img_p in enumerate(img_paths, start=1):
    img = cv2.imread(str(img_p))
    if img is None:
        continue
    h, w = img.shape[:2]
    det_res = det_model.predict(str(img_p), conf=CONF_THRES, iou=0.55, imgsz=640, verbose=False)[0]

    persons, bikes, others = [], [], []
    if det_res.boxes is not None and len(det_res.boxes) > 0:
        bboxes = det_res.boxes.xyxy.cpu().numpy()
        cls_ids = det_res.boxes.cls.cpu().numpy().astype(int)
        sam_res = sam_model.predict(str(img_p), bboxes=bboxes, verbose=False)[0]
        if sam_res.masks is not None:
            masks_np = sam_res.masks.data.cpu().numpy()
            for m, pid in zip(masks_np, cls_ids):
                m_bin = cv2.resize((m > 0.5).astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
                comp_cls = PROMPT_GROUPS[pid][1]
                if comp_cls == 8:
                    persons.append(m_bin)
                elif comp_cls in (4, 6):
                    bikes.append((comp_cls, m_bin))
                else:
                    others.append((comp_cls, m_bin))

    used_p = set()
    final_masks = []
    for b_cls, b_mask in bikes:
        matched = [pi for pi, pm in enumerate(persons) if pi not in used_p and masks_overlap(pm, b_mask)]
        if matched:
            merged = b_mask.copy()
            for pi in matched:
                used_p.add(pi)
                merged = np.logical_or(merged > 0, persons[pi] > 0).astype(np.uint8)
            final_masks.append((5 if b_cls == 4 else 7, merged))
        else:
            final_masks.append((b_cls, b_mask))

    for pi, pm in enumerate(persons):
        if pi not in used_p:
            final_masks.append((8, pm))
    final_masks.extend(others)

    lines = []
    for cid, m_bin in final_masks:
        poly = mask_to_norm_poly(m_bin, w, h)
        if poly is None:
            continue
        lines.append(f"{cid} " + " ".join(f"{x:.6f} {y:.6f}" for x, y in poly))
        stats[CLASSES[cid]] += 1

    shutil.copy2(img_p, img_out / img_p.name)
    (lbl_out / f"{img_p.stem}.txt").write_text("\n".join(lines), encoding="utf-8")
    if idx % 10 == 0 or idx == len(img_paths):
        print(f"Processed {idx}/{len(img_paths)} images...")

zip_path = shutil.make_archive("/content/sam2_prelabeled", "zip", root_dir=out_dir)
print("\n=== SAM 2.1 + YOLO-WORLD SUMMARY ===")
print(dict(stats))
print("Saved:", zip_path)

In [ ]:
#@title 4. Tải file sam2_prelabeled.zip về máy (để import CVAT hoặc gộp vào team780)
from google.colab import files
files.download("/content/sam2_prelabeled.zip")